# UPLOADING OF DATASET

In [57]:
import pandas as pd
data=pd.read_csv('/kaggle/input/datasets/marcyaquaks/water-assessment/water_quality_dataset1.csv')
data


,pH,TDS_ppm,Turbidity_NTU,Temperature_C,Label
0,7.38,138.7,0.25,20.25,Safe
1,7.42,79.2,0.45,20.14,Safe
2,7.05,104.1,1.18,20.12,Safe
3,7.32,112.8,0.53,19.65,Safe
4,6.87,171.9,1.97,19.42,Safe
...,...,...,...,...,...
100795,7.85,1150.7,28.62,25.84,Unsafe
100796,8.28,993.4,25.85,26.00,Unsafe
100797,8.09,1105.9,25.40,26.86,Unsafe
100798,8.16,1060.5,28.47,26.62,Unsafe


# Understanding the dataset

In [58]:
data.info


<bound method DataFrame.info of           pH  TDS_ppm  Turbidity_NTU  Temperature_C   Label
0       7.38    138.7           0.25          20.25    Safe
1       7.42     79.2           0.45          20.14    Safe
2       7.05    104.1           1.18          20.12    Safe
3       7.32    112.8           0.53          19.65    Safe
4       6.87    171.9           1.97          19.42    Safe
...      ...      ...            ...            ...     ...
100795  7.85   1150.7          28.62          25.84  Unsafe
100796  8.28    993.4          25.85          26.00  Unsafe
100797  8.09   1105.9          25.40          26.86  Unsafe
100798  8.16   1060.5          28.47          26.62  Unsafe
100799  8.23   1087.0          34.08          26.11  Unsafe

[100800 rows x 5 columns]>

In [59]:
data.shape


(100800, 5)

In [60]:
data.describe()

,pH,TDS_ppm,Turbidity_NTU,Temperature_C
count,100800.000000,100800.000000,100800.000000,100800.000000
mean,7.593915,692.646992,11.306750,24.883516
std,0.799464,647.331906,9.868407,2.688138
min,6.090000,28.600000,0.000000,16.560000
25%,7.010000,163.000000,1.920000,23.090000
50%,7.380000,472.850000,9.970000,25.040000
75%,8.100000,1023.325000,18.540000,26.930000
max,9.960000,2806.100000,72.750000,31.510000


# Data Cleaning and preprocessing

In [61]:
data.isnull().sum() #checking missing values

pH               0
TDS_ppm          0
Turbidity_NTU    0
Temperature_C    0
Label            0
dtype: int64

In [62]:
print(data.drop_duplicates(inplace=True)) #checking for duplicates

None


In [63]:
data['Label'].unique

<bound method Series.unique of 0           Safe
1           Safe
2           Safe
3           Safe
4           Safe
           ...  
100795    Unsafe
100796    Unsafe
100797    Unsafe
100798    Unsafe
100799    Unsafe
Name: Label, Length: 100800, dtype: object>

In [64]:
data.info

<bound method DataFrame.info of           pH  TDS_ppm  Turbidity_NTU  Temperature_C   Label
0       7.38    138.7           0.25          20.25    Safe
1       7.42     79.2           0.45          20.14    Safe
2       7.05    104.1           1.18          20.12    Safe
3       7.32    112.8           0.53          19.65    Safe
4       6.87    171.9           1.97          19.42    Safe
...      ...      ...            ...            ...     ...
100795  7.85   1150.7          28.62          25.84  Unsafe
100796  8.28    993.4          25.85          26.00  Unsafe
100797  8.09   1105.9          25.40          26.86  Unsafe
100798  8.16   1060.5          28.47          26.62  Unsafe
100799  8.23   1087.0          34.08          26.11  Unsafe

[100800 rows x 5 columns]>

In [65]:
data['Label']=data['Label'].map({
    'Safe':1,
    'Unsafe':0
})
data['Turbidity_NTU'] = data['Turbidity_NTU'].apply(lambda x: 1 if x >= 5 else 0)

In [66]:
data.info

<bound method DataFrame.info of           pH  TDS_ppm  Turbidity_NTU  Temperature_C  Label
0       7.38    138.7              0          20.25      1
1       7.42     79.2              0          20.14      1
2       7.05    104.1              0          20.12      1
3       7.32    112.8              0          19.65      1
4       6.87    171.9              0          19.42      1
...      ...      ...            ...            ...    ...
100795  7.85   1150.7              1          25.84      0
100796  8.28    993.4              1          26.00      0
100797  8.09   1105.9              1          26.86      0
100798  8.16   1060.5              1          26.62      0
100799  8.23   1087.0              1          26.11      0

[100800 rows x 5 columns]>

In [67]:
print(data['Label'].value_counts())

Label
0    70560
1    30240
Name: count, dtype: int64


## Selcecting number of rows to work on

In [68]:
# Separate the two classes
safe = data[data['Label'] == 1]
unsafe = data[data['Label'] == 0]

# Randomly select 500 from each class
safe_sample = safe.sample(n=3000, random_state=42)
unsafe_sample = unsafe.sample(n=7000, random_state=42)

# Combine them
small_data = pd.concat([safe_sample, unsafe_sample])

# Shuffle the dataset
small_data = small_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Check the result
print(small_data['Label'].value_counts())

Label
0    7000
1    3000
Name: count, dtype: int64


## Defining columns into feature and Target

In [69]:
X = small_data[['pH', 'TDS_ppm', 'Turbidity_NTU', 'Temperature_C']]
y = small_data['Label']

## Splittind Dataset 

In [70]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [71]:
print("Training labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training labels:
Label
0    5600
1    2400
Name: count, dtype: int64

Testing labels:
Label
0    1400
1     600
Name: count, dtype: int64


# Standardization

In [72]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on the training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler on the test data
X_test_scaled = scaler.transform(X_test)

In [73]:
print(X_train.columns)

Index(['pH', 'TDS_ppm', 'Turbidity_NTU', 'Temperature_C'], dtype='object')


## Traiing The Logistic Regression model

In [74]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42)

## Model Prediction

In [75]:

y_pred = model.predict(X_test_scaled)

# Model Evaluation

In [76]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print()

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))
print()

print("Classification Report")
print(classification_report(y_test, y_pred))

Accuracy: 0.9975

Confusion Matrix
[[1395    5]
 [   0  600]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1400
           1       0.99      1.00      1.00       600

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



## FIXED INPUT FORMAT

In [77]:
def get_user_input():
    ph = float(input("Enter pH: "))
    tds = float(input("Enter TDS (ppm): "))
    turbidity = float(input("Enter Turbidity (NTU): "))
    temperature = float(input("Enter Temperature (°C): "))

    return ph, tds, turbidity, temperature

## Machine Learning Prediction

In [78]:
import pandas as pd

def ml_prediction(ph, tds, turbidity, temperature):

    new_sample = pd.DataFrame(
        [[ph, tds, turbidity, temperature]],
        columns=scaler.feature_names_in_
    )

    new_scaled = scaler.transform(new_sample)

    pred = model.predict(new_scaled)[0]
    prob = model.predict_proba(new_scaled)[0]

    return pred, prob

## Calculation of Purity Score 

In [79]:
def calculate_purity(ph, tds, turbidity, temperature):

    # pH score
    ph_score = 100 if 6.5 <= ph <= 8.5 else max(0, 100 - abs(7.5 - ph) * 20)

    # TDS score (WHO: 500 ppm)
    tds_score = max(0, 100 - (tds / 500) * 100)

    # Turbidity score (WHO: 5 NTU)
    turb_score = max(0, 100 - (turbidity / 5) * 100)

    # Temperature score (ideal 20–30)
    temp_score = 100 if 20 <= temperature <= 30 else max(0, 100 - abs(25 - temperature) * 5)

    # Weighted WQI
    purity = (
        0.30 * ph_score +
        0.30 * tds_score +
        0.30 * turb_score +
        0.10 * temp_score
    )

    return purity

In [80]:
def classify_water(purity):

    if purity >= 90:
        return "Excellent"
    elif purity >= 75:
        return "Good"
    elif purity >= 50:
        return "Fair"
    elif purity >= 25:
        return "Poor"
    else:
        return "Very Poor"

# Treatment Recommendation

In [81]:
def recommend_treatment(ph, tds, turbidity, purity):

    treatment = []

    if turbidity > 5:
        treatment.append("Coagulation + Filtration")

    if tds > 500:
        treatment.append("Reverse Osmosis / Ion Exchange")

    if ph < 6.5:
        treatment.append("Lime dosing (raise pH)")
    elif ph > 8.5:
        treatment.append("Acid neutralization")

    if purity >= 90:
        treatment.append("Disinfection only (UV/Chlorine)")
    elif purity >= 75:
        treatment.append("Filtration + Disinfection")
    elif purity >= 50:
        treatment.append("Advanced filtration required")
    else:
        treatment.append("Full multi-stage treatment system")

    return treatment

# Decision Confusion

In [82]:
#Combining ML prediction + Engineering purity to produce one final intelligent decision
def decision_fusion(ml_pred, ml_prob, purity):

    ml_safe_prob = ml_prob[1]
    ml_unsafe_prob = ml_prob[0]

    # Decision logic
    if ml_pred == 1 and purity >= 75:
        decision = "SAFE (High Confidence)"
        agreement = 1

    elif ml_pred == 0 and purity < 50:
        decision = "UNSAFE (High Confidence)"
        agreement = 1

    else:
        decision = "UNCERTAIN - Further Testing Required"
        agreement = 0

    # Confidence score
    confidence = (ml_safe_prob * 0.5) + (purity / 100 * 0.5)

    return decision, confidence, agreement

## FINAL SYSTEM

In [83]:
ph, tds, turbidity, temperature = get_user_input()

# ML
ml_pred, ml_prob = ml_prediction(ph, tds, turbidity, temperature)

# Engineering
purity = calculate_purity(ph, tds, turbidity, temperature)
classification = classify_water(purity)
treatment = recommend_treatment(ph, tds, turbidity, purity)

# Fusion
final_decision, confidence, agreement = decision_fusion(
    ml_pred,
    ml_prob,
    purity
)

# OUTPUT
print("\n================ WATER AI SYSTEM ================\n")

print(f"pH: {ph}")
print(f"TDS: {tds}")
print(f"Turbidity: {turbidity}")
print(f"Temperature: {temperature}")

print("\n--- MACHINE LEARNING ---")
print("Prediction:", "SAFE" if ml_pred == 1 else "UNSAFE")
print(f"Safe Probability: {ml_prob[1]*100:.2f}%")

print("\n--- ENGINEERING WQI ---")
print(f"Purity Index: {purity:.2f}%")
print("Class:", classification)

print("\n--- FINAL AI DECISION ---")
print("Decision:", final_decision)
print(f"Confidence Score: {confidence*100:.2f}%")

print("\n--- TREATMENT PLAN ---")
for t in treatment:
    print("-", t)

Enter pH:  7
Enter TDS (ppm):  132
Enter Turbidity (NTU):  3
Enter Temperature (°C):  22



================ WATER AI SYSTEM ================

pH: 7.0
TDS: 132.0
Turbidity: 3.0
Temperature: 22.0

--- MACHINE LEARNING ---
Prediction: UNSAFE
Safe Probability: 0.00%

--- ENGINEERING WQI ---
Purity Index: 74.08%
Class: Fair

--- FINAL AI DECISION ---
Decision: UNCERTAIN - Further Testing Required
Confidence Score: 37.04%

--- TREATMENT PLAN ---
- Advanced filtration required
